In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Zeitspanne auswählen
dates = pd.date_range("2025-11-05", "2025-11-12")

# ✅ Hier manuell deine Task-Verläufe eintragen
remaining_tasks = [5, 5, 5, 4, 2, 2, 1,1]

# Anzahl Gesamt-Tasks aus erstem Wert ableiten
tasks_total = remaining_tasks[0]

# DataFrame
data = pd.DataFrame({
    "Date": dates,
    "Remaining Tasks": remaining_tasks
})

plt.figure(figsize=(8, 5))
plt.plot(data["Date"], data["Remaining Tasks"], marker="o", linewidth=2, label="Tatsächlicher Verlauf")

plt.title("Burndown Chart (Nov 05 – Nov 12, 2025)")
plt.xlabel("Datum")
plt.ylabel("Offene Tasks")

# ✅ Y-Achse mit extra Höhe
plt.yticks(range(0, tasks_total + 2))
plt.ylim(0, tasks_total + 1)

plt.xticks(rotation=45)
plt.grid(True, linestyle="--", alpha=0.6)

# ✅ Ideale Burndown-Linie (automatisch berechnet)
ideal_line = [tasks_total - (tasks_total / (len(dates)-1)) * i for i in range(len(dates))]
plt.plot(data["Date"], ideal_line, linestyle="--", linewidth=1.8, label="Ideale Linie")

plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from datetime import datetime, timedelta

# --- Datengrundlage ---
assignments = [
    ("Amine", "Weekly Meeting", "2025-11-05 16:30", "2025-11-05 17:15"),
    ("Andree", "Weekly Meeting", "2025-11-05 16:30", "2025-11-05 17:15"),
    ("Felix", "Weekly Meeting", "2025-11-05 16:30", "2025-11-05 17:15"),
    ("Moritz", "Weekly Meeting", "2025-11-05 16:30", "2025-11-05 17:15"),
    ("Sofian", "Weekly Meeting", "2025-11-05 16:30", "2025-11-05 17:15"),
    ("Yassine", "Weekly Meeting", "2025-11-05 16:30", "2025-11-05 17:15"),
    ("Felix", "Fix Issues", "2025-11-06 15:30", "2025-11-06 16:00"),
    ("Felix", "Organisation", "2025-11-06 16:00", "2025-11-06 16:30"),
    ("Sofian", "Grundgerüst überarbeiten", "2025-11-07 15:00", "2025-11-07 17:15"),
    ("Jordan", "Grundgerüst überarbeiten", "2025-11-07 15:00", "2025-11-07 17:15"),
    ("Andree", "UserStories Akzeptanzkriterien", "2025-11-08 15:30", "2025-11-08 16:00"),
    ("Yassine", "Backend Arbeit", "2025-11-08 13:00", "2025-11-08 16:00"),
    ("Moritz", "CSS-Stylesheet", "2025-11-11 10:30", "2025-11-11 11:30"),
    ("Moritz", "CSS-Hompage & Content", "2025-11-11 11:30", "2025-11-11 12:30"),
    ("Amine", "CSS-Stylesheet", "2025-11-11 10:30", "2025-11-11 11:30"),
    ("Amine", "CSS-Hompage & Content", "2025-11-11 11:30", "2025-11-11 12:30"),
    ("Felix", "Präsentation vorbereiten", "2025-11-11 19:30", "2025-11-11 20:45"),
    ("Moritz", "Präsentation vorbereiten", "2025-11-11 19:30", "2025-11-11 20:45"),
]

df = pd.DataFrame(assignments, columns=["employee", "task", "start", "end"])
df["start"] = pd.to_datetime(df["start"])
df["end"] = pd.to_datetime(df["end"])

# --- Mitarbeiter chronologisch sortieren ---
first_tasks = df.groupby("employee")["start"].min().sort_values()
employees = first_tasks.index.tolist()
y_pos = {emp: i for i, emp in enumerate(employees)}

# --- Farben pro Task ---
colors = list(mcolors.TABLEAU_COLORS.values())
task_colors = {task: colors[i % len(colors)] for i, task in enumerate(df["task"].unique())}

# --- Plot ---
plt.style.use("seaborn-v0_8-whitegrid")
fig, ax = plt.subplots(figsize=(14, 6))

# --- Hintergrund einfärben: ganze Tage ---
start_date = df["start"].dt.floor("D").min()
end_date = df["end"].dt.ceil("D").max()

curr_date = start_date
while curr_date < end_date:
    next_day = curr_date + timedelta(days=1)
    day_start = datetime.combine(curr_date, datetime.min.time())
    day_end = datetime.combine(next_day, datetime.min.time())

    if curr_date.weekday() >= 5:
        color = "#ffeaea"  # Wochenende
    else:
        color = "#dcdcdc" if curr_date.day % 2 == 0 else "#eaeaea"

    ax.axvspan(mdates.date2num(day_start), mdates.date2num(day_end),
               facecolor=color, alpha=0.9, zorder=0)
    curr_date = next_day

# --- Vertikale Linien jede 6 Stunden ---
hour_lines = pd.date_range(start=start_date, end=end_date, freq="6H")
for h in hour_lines:
    ax.axvline(mdates.date2num(h), color="lightgray", linestyle="--", linewidth=0.6, zorder=1)

# --- Gantt-Balken ---
for _, r in df.iterrows():
    start_num = mdates.date2num(r["start"])
    end_num = mdates.date2num(r["end"])
    width = end_num - start_num
    y = y_pos[r["employee"]]

    ax.barh(y, width, left=start_num, height=0.45,
            color=task_colors[r["task"]], edgecolor="black", linewidth=0.8, zorder=2)

# --- Achsen & Formatierung ---
ax.set_yticks(list(y_pos.values()))
ax.set_yticklabels(list(y_pos.keys()), fontsize=10)
ax.xaxis_date()
ax.xaxis.set_major_locator(mdates.DayLocator())
ax.xaxis.set_minor_locator(mdates.HourLocator(interval=6))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m\n%a'))
ax.set_xlabel("Datum / Uhrzeit", fontsize=10)

ax.set_xlim([
    mdates.date2num(datetime.combine(start_date, datetime.min.time())),
    mdates.date2num(datetime.combine(end_date, datetime.min.time()))
])

ax.set_title(f"Gantt-Diagramm: {start_date:%d.%m.%Y} – {end_date:%d.%m.%Y}",
             fontsize=12, weight="bold")

# --- Legende ---
patches = [mpatches.Patch(color=c, label=t) for t, c in task_colors.items()]
legend = ax.legend(handles=patches, title="Tasks", loc='upper left',
                   bbox_to_anchor=(1.02, 1), frameon=True)

# ==========================
#  ARBEITSSTUNDEN-BERECHNUNG
# ==========================

df["hours"] = (df["end"] - df["start"]).dt.total_seconds() / 3600

# Gemeinsame Stunden werden allen Mitarbeitenden angerechnet
individual_hours = df.groupby("employee")["hours"].sum().reset_index()
individual_hours.columns = ["Mitarbeiter", "Stunden"]
individual_hours["Stunden"] = individual_hours["Stunden"].round(2)
gesamtstunden = individual_hours["Stunden"].sum().round(2)

# --- Tabelle dynamisch unter der Legende ---
fig.canvas.draw()  # notwendig, damit die Legendenposition berechnet wird
legend_box = legend.get_window_extent()
inv = fig.transFigure.inverted()
legend_box_fig = inv.transform(legend_box)
legend_x0, legend_y0, legend_x1, legend_y1 = legend_box_fig.flatten()

table_y = legend_y0 - 0.02  # kleiner Abstand unterhalb der Legende

table = plt.table(
    cellText=individual_hours.values,
    colLabels=individual_hours.columns,
    cellLoc='center',
    colLoc='center',
    loc='upper left',
    bbox=[1.02, table_y - 0.35, 0.20, 0.35]  # dynamische Position
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.1, 1.3)

# --- Gesamtstunden unter der Tabelle ---
projekt_stunden_bisher = 57.5  # ⬅️ HIER deinen bisherigen Projektstand eintragen
gesamtstunden_projekt = projekt_stunden_bisher + gesamtstunden

# Textanzeige
plt.text(
    1.02, table_y - 0.40,
    f"Gesamtstunden (Woche): {gesamtstunden:.1f} h",
    transform=ax.transAxes,
    fontsize=11,
    fontweight='bold',
)

plt.text(
    1.02, table_y - 0.47,  # etwas tiefer platziert
    f"Gesamtstunden (Projekt): {gesamtstunden_projekt:.1f} h",
    transform=ax.transAxes,
    fontsize=11,
    fontweight='bold',
)
